# Microsoft Agent Framework — Azure OpenAI (Responses API)

I denne kodeeksempel vil du bruge **Microsoft Agent Framework (MAF)** til at oprette en simpel agent understøttet af **Azure OpenAI** ved hjælp af **Responses API**.

> **Migreringsnote:** Dette eksempel brugte tidligere Semantic Kernel med GitHub Models. Det er blevet migreret til Microsoft Agent Framework, og GitHub Models (forældet, udfases juli 2026) er blevet erstattet med Azure OpenAI, som understøtter Responses API. `OpenAIChatClient` i MAF retter sig mod Azure OpenAIs stabile `/openai/v1/` endpoint og bruger som standard Responses API.

Formålet med dette eksempel er at demonstrere de trin, som senere vil blive anvendt i yderligere kodeeksempler ved implementering af forskellige agent-mønstre.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importer de nødvendige Python-pakker


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definere et Værktøj

I Microsoft Agent Framework er et **værktøj** en simpel Python-funktion dekoreret med `@tool`, som agenten kan kalde. Nedenfor definerer vi et værktøj, der returnerer en tilfældig feriedestination og undgår at gentage den forrige.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Opret agenten

Her opretter vi agenten med navnet `TravelAgent`.

I dette eksempel bruger vi meget enkle instruktioner. Du er velkommen til at ændre disse instruktioner for at se, hvordan agentens adfærd ændrer sig.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Kørsel af Agenten

Nu kan vi køre agenten. Vi opretter en `AgentSession`, så agenten husker samtalen på tværs af omgange, og sender derefter to `user_inputs`. Den første beder om en rejse; den anden siger, at brugeren ikke kunne lide forslaget og beder om et andet — agenten bruger sessionshistorikken plus værktøjet `get_random_destination` til at svare.

Du kan ændre disse beskeder for at se, hvordan agenten reagerer forskelligt. Svar bliver **streamet** token-for-token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
